# Assignment — Bloc 5a, Sessió 8
### Mini-repte: Construeix el teu Oscillator, Envelope i SamplePlayer

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

**Objectiu:** Completa les tres classes marcades amb `# TODO`. Cada part té una cel·la d'autotest.

No esborris les cel·les d'autotest.

In [ ]:
import numpy as np
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import urllib.request

sample_rate = 44100

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/05_sintesi/sessio08_oscil_ladors_classes/recursos/audio/"
urllib.request.urlretrieve(base_url + "kick_sample.wav", "kick_sample.wav")
print("Fitxers carregats")

## Part 1 — La classe `Oscillator`

Completa `generate`: ha de retornar un array amb la forma d'ona indicada (`sine`, `square` o `sawtooth`).

In [ ]:
class Oscillator:
    def __init__(self, freq=440.0, waveform='sine', sample_rate=44100):
        self.freq = freq
        self.waveform = waveform
        self.sample_rate = sample_rate

    def set_freq(self, freq):
        # TODO 1: actualitza self.freq
        pass

    def generate(self, duration):
        n = int(self.sample_rate * duration)
        t = np.linspace(0, duration, n, endpoint=False)

        # TODO 2: retorna l'ona corresponent a self.waveform
        # 'sine'     -> np.sin(2*pi*self.freq*t)
        # 'square'   -> signal.square(2*pi*self.freq*t)
        # 'sawtooth' -> signal.sawtooth(2*pi*self.freq*t)
        return None  # <-- substitueix

In [ ]:
# AUTOTEST 1
osc_test = Oscillator(freq=440, waveform='sine')
wave_test = osc_test.generate(duration=1.0)

assert wave_test is not None, "generate() retorna None"
assert len(wave_test) == 44100, f"S'esperaven 44100 mostres, n'hi ha {len(wave_test)}"
assert np.max(np.abs(wave_test)) > 0.9, "L'amplitud sembla incorrecta per a un sinus pur"

osc_test.set_freq(880)
assert osc_test.freq == 880, "set_freq() no actualitza self.freq"

osc_square = Oscillator(freq=220, waveform='square')
wave_square = osc_square.generate(duration=0.5)
assert wave_square is not None
assert len(wave_square) == 22050

print("✅ Part 1 correcta!")
display(Audio(wave_test, rate=sample_rate))

## Part 2 — La classe `Envelope` (ADSR)

Completa `generate`: ha de retornar la corba ADSR completa (Attack + Decay + Sustain + Release) com un sol array.

In [ ]:
class Envelope:
    def __init__(self, attack=0.01, decay=0.1, sustain=0.7, release=0.2):
        self.attack = attack
        self.decay = decay
        self.sustain = sustain
        self.release = release

    def generate(self, note_duration, sample_rate=44100):
        n_attack  = int(self.attack  * sample_rate)
        n_decay   = int(self.decay   * sample_rate)
        n_sustain = max(0, int(note_duration * sample_rate) - n_attack - n_decay)
        n_release = int(self.release * sample_rate)

        # TODO 3: crea les 4 fases:
        # env_attack:  np.linspace(0, 1, n_attack)
        # env_decay:   np.linspace(1, self.sustain, n_decay)
        # env_sustain: np.full(n_sustain, self.sustain)
        # env_release: np.linspace(self.sustain, 0, n_release)
        env_attack = None   # <-- substitueix
        env_decay = None    # <-- substitueix
        env_sustain = None  # <-- substitueix
        env_release = None  # <-- substitueix

        # TODO 4: concatena les 4 fases en un sol array (np.concatenate)
        return None  # <-- substitueix

In [ ]:
# AUTOTEST 2
env_test = Envelope(attack=0.01, decay=0.1, sustain=0.7, release=0.2)
curve = env_test.generate(note_duration=0.5)

assert curve is not None, "generate() retorna None"
assert curve[0] < 0.05, f"L'inici hauria de ser ~0, és {curve[0]:.3f}"
assert abs(curve[-1]) < 0.05, f"El final hauria de ser ~0, és {curve[-1]:.3f}"

# El punt mitjà del sustain hauria d'estar prop de 0.7
mig = len(curve) // 2
assert abs(curve[mig] - 0.7) < 0.1, \
    f"El sustain hauria d'estar prop de 0.7, és {curve[mig]:.3f}"

print("✅ Part 2 correcta!")
plt.figure(figsize=(10,3))
plt.plot(curve)
plt.title("La teva envolvent ADSR")
plt.show()

## Part 3 — Combinar Oscillator + Envelope

Completa `tocar_nota`: rep un número MIDI i una durada, i retorna l'array d'una nota completa (oscil·lador × envolvent).

In [ ]:
def note_to_freq(note_number):
    """Converteix número MIDI a freqüència (Hz). Ja proporcionada."""
    return 440.0 * (2 ** ((note_number - 69) / 12))


def tocar_nota(note_number, note_duration, waveform='sine',
                attack=0.02, decay=0.1, sustain=0.6, release=0.2):
    # TODO 5: converteix note_number a freqüència amb note_to_freq
    freq = None  # <-- substitueix

    # TODO 6: crea un Oscillator i un Envelope amb els paràmetres rebuts
    osc = None  # <-- Oscillator(freq=freq, waveform=waveform)
    env = None  # <-- Envelope(attack=attack, decay=decay, sustain=sustain, release=release)

    # TODO 7: genera l'ona i l'envolvent, i multiplica-les
    # (la durada de l'oscil·lador ha de ser note_duration + release)
    total_duration = note_duration + release
    wave = None       # <-- osc.generate(total_duration)
    env_curve = None  # <-- env.generate(note_duration)

    n = min(len(wave), len(env_curve))
    return wave[:n] * env_curve[:n]

In [ ]:
# AUTOTEST 3
nota_test = tocar_nota(60, note_duration=0.3, waveform='sine')

assert nota_test is not None, "tocar_nota() retorna None"
assert len(nota_test) > 0, "L'array resultant és buit"
assert nota_test[0] < 0.05, "L'inici hauria de començar fluix (Attack)"
assert abs(nota_test[-1]) < 0.05, "El final hauria d'acabar en silenci (Release)"
assert np.max(np.abs(nota_test)) > 0.3, "L'amplitud màxima sembla massa baixa"

print("✅ Part 3 correcta!")
display(Audio(nota_test, rate=sample_rate))

## Part 4 — La classe `SamplePlayer`

Completa `play`: ha de retornar el sample carregat, multiplicat per `gain`.

In [ ]:
class SamplePlayer:
    def __init__(self, filepath, sample_rate=44100):
        self.sample_rate = sample_rate
        data, sr = sf.read(filepath)
        self.sample = data

    def play(self, gain=1.0):
        # TODO 8: retorna self.sample multiplicat per gain
        return None  # <-- substitueix

In [ ]:
# AUTOTEST 4
kick = SamplePlayer('kick_sample.wav')
audio_normal = kick.play(gain=1.0)
audio_fluix = kick.play(gain=0.3)

assert audio_normal is not None, "play() retorna None"
assert len(audio_normal) == len(kick.sample)
assert np.isclose(np.max(np.abs(audio_fluix)), np.max(np.abs(audio_normal)) * 0.3, atol=0.01), \
    "El gain no s'aplica correctament"

print("✅ Part 4 correcta!")
display(Audio(audio_normal, rate=sample_rate))

## Part 5 — Crea la teva pròpia seqüència

Combina les classes que has construït: crea una seqüència d'almenys 6 notes (amb `tocar_nota`) que incloga almenys un cop de `SamplePlayer` (el kick) en algun punt.

In [ ]:
# TODO 9: crea la teva seqüència
# Pots concatenar trossos amb np.concatenate([tros1, tros2, ...])
# o sumar-los si vols que sonin simultanis (com a Sessió 2: mixing)

la_meva_sequencia = np.array([])  # <-- substitueix

Audio(la_meva_sequencia, rate=sample_rate)

In [ ]:
# AUTOTEST 5
assert len(la_meva_sequencia) > sample_rate * 0.5, \
    "La seqüència sembla massa curta — assegura't que hi ha almenys 6 notes"

print("✅ Part 5 correcta! Escolta el resultat a la cel·la anterior.")

---
## 🚀 Challenges (opcional)

1. **Polifonia simple:** crea una llista de 3-4 `Oscillator` amb freqüències diferents (un acord) i suma'ls per sentir-los simultàniament.
2. **Envolvent percussiva:** crea una `Envelope` amb `attack` molt curt (0.001) i `sustain=0` — com sona? S'assembla a algun instrument?
3. **Vibrato:** modifica `Oscillator.generate` perquè la freqüència oscil·li lleugerament al llarg del temps (un LFO sobre `self.freq`).
4. **El teu propi "kit":** desa 2-3 samples curts diferents (pots generar-los com a la Sessió 2) i crea una classe `Kit` que en guardi diversos i els pugui disparar per nom.

In [ ]:
# El teu codi del Challenge aquí